# Face-Enabled Multipart Infill Evaluation

This notebook checks the face codec floor, face infill quality, and body-motion retention after appending face as the fifth 512x4 RVQ stream. The old and new transformers are evaluated on the same face-available validation clips.

In [ ]:
from pathlib import Path
import gc
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Video, display

cwd = Path.cwd().resolve()
if (cwd / 'motion_generation').is_dir():
    PROJECT_ROOT = cwd
elif cwd.name == 'notebooks' and cwd.parent.name == 'motion_generation':
    PROJECT_ROOT = cwd.parents[1]
else:
    raise RuntimeError(f'Run from the repository root or motion_generation/notebooks, not {cwd}')

MOTION_GENERATION_DIR = PROJECT_ROOT / 'motion_generation'
if str(MOTION_GENERATION_DIR) not in sys.path:
    sys.path.insert(0, str(MOTION_GENERATION_DIR))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from scripts.train_audio_mask_multipart import discover_names, load_sequences, read_split_file
from utils.face_infill_visualization import (
    plot_tiled_full_clip_summary,
    prepare_tiled_full_clip_comparison,
    save_tiled_full_clip_video,
    select_representative_clip_row,
)
from utils.multipart_motion import MULTIMODAL_PART_ORDER, PART_ORDER
from utils.variable_c2f_evaluation import (
    InfillModelSpec,
    compute_fid_table,
    compute_retrieval_table,
    evaluate_face_codec_reconstruction,
    evaluate_model_windows,
    export_full_clip_gap,
    load_audio_motion_transformer,
    load_part_codecs,
    summarize_window_metrics,
)

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

In [ ]:
DATA_DIR = PROJECT_ROOT / 'SuSuInterActs' / 'SuSuInterActs'
AUDIO_DIR = DATA_DIR / 'audio_features_hubert_layer9_fps10'
BODY_TOKEN_DIR = DATA_DIR / 'motion_token_data_multipart_512x4'
FACE_TOKEN_DIR = DATA_DIR / 'motion_face_token_data_multipart_512x4'
EVAL_SPLIT = DATA_DIR / 'split' / 'val_file_list.txt'
FACE_DIR = DATA_DIR / 'arkit_data'
MOTION_DIR = DATA_DIR / 'motion_data'
WAV_DIR = DATA_DIR / 'wav_data'
MOTION2TEXT_JSON = DATA_DIR / 'text_data' / 'motion2text.json'
CHECKPOINT_ROOT = PROJECT_ROOT / 'checkpoints'
RVQ_ROOT = CHECKPOINT_ROOT / 'multipart_rvqvae'

BODY_MODEL = InfillModelSpec(
    name='soft_recovery_body_only',
    checkpoint=CHECKPOINT_ROOT / 'mask_multipart_variable_c2f_soft_recovery_sf05_gap1_15',
    decoder='c2f',
    allowed_gaps=tuple(range(1, 16)),
)
FACE_STAGE1_MODEL = InfillModelSpec(
    name='face_fixed_targets_no_sf_stage1',
    checkpoint=CHECKPOINT_ROOT / 'mask_multipart_face_variable_c2f_fixed_targets_no_sf_scratch_gap1_15',
    decoder='c2f',
    allowed_gaps=tuple(range(1, 16)),
)
FACE_MODEL = InfillModelSpec(
    name='face_soft_recovery_sf05_stage2',
    checkpoint=CHECKPOINT_ROOT / 'mask_multipart_face_variable_c2f_soft_recovery_sf05_stage2_gap1_15',
    decoder='c2f',
    allowed_gaps=tuple(range(1, 16)),
)

BODY_CHECKPOINTS = {
    part: RVQ_ROOT / f'rvq_{part}_512x4_bs256_cosine' / 'model' / 'best.pth'
    for part in PART_ORDER
}
FACE_CHECKPOINTS = {
    **BODY_CHECKPOINTS,
    'face': RVQ_ROOT / 'rvq_face_512x4_bs256_cosine' / 'model' / 'best.pth',
}
GAPS = [3, 7, 15]
WINDOWS_PER_CLIP = 1
BATCH_SIZE = 32
MAX_EVAL_CLIPS = None
OUT_DIR = MOTION_GENERATION_DIR / 'notebooks' / 'face_infill_eval_outputs'

EVALUATION_DIR = PROJECT_ROOT / 'evaluation'
EVALUATOR_CKPT = CHECKPOINT_ROOT / 'eval_model' / 'best_model.pt'
EVALUATOR_CONFIG = EVALUATION_DIR / 'config' / 'train_bert_orig.yaml'
EVALUATOR_STATS_DIR = EVALUATION_DIR / 'stats' / 'humanml3d' / 'guoh3dfeats'
EVALUATOR_TEXT_MODEL = os.environ.get('EVALUATOR_TEXT_MODEL_PATH', 'bert-base-chinese')

RUN_FULL_CLIP_EXPORT = False
RUN_FID_DIVERSITY = False
RUN_RETRIEVAL_RK = False
RUN_FULL_CLIP_VISUALIZATIONS = False
VISUALIZATION_GAPS = [3, 7, 15]
VISUALIZATION_FRAME_STEP = 2
TEMPLATE_BVH = MOTION_GENERATION_DIR / 'meta' / 'template_susu_retarget_63nodes.bvh'

## Full-Clip Motion Export

Set `RUN_FULL_CLIP_EXPORT=True` and run this cell once. The old body model uses its 16-slot tokens; Stage 1 and Stage 2 use their 20-slot body+face tokens. All three use the same paired clip order and raw-motion ground truth.

In [ ]:
FULL_CLIP_DIR = OUT_DIR / 'full_clips'
def run_full_clip_export():
    FULL_RUNS = [
        (BODY_MODEL, body_sequences, body_codecs),
        (FACE_STAGE1_MODEL, face_sequences, face_codecs),
        (FACE_MODEL, face_sequences, face_codecs),
    ]
    manifests = []
    for gap in GAPS:
        for spec, sequences, codecs in FULL_RUNS:
            print(f'exporting {spec.name}, gap={gap}')
            model = load_audio_motion_transformer(spec.checkpoint, DEVICE)
            manifests.append(export_full_clip_gap(
                model, spec, sequences, codecs,
                gap=gap, batch_size=BATCH_SIZE, device=DEVICE,
                output_root=FULL_CLIP_DIR, motion_dir=MOTION_DIR,
                gt_source='raw_motion_data', canonicalize_raw_root=False,
                slot_constrained=False, clean=True,
            ))
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    full_manifest = pd.concat(manifests, ignore_index=True)
    full_manifest.to_csv(OUT_DIR / 'full_clip_manifest.csv', index=False)
    display(full_manifest.groupby(['model', 'gap']).size().rename('clips'))

## Full-Clip FID and Diversity

In [ ]:
MODEL_NAMES = [BODY_MODEL.name, FACE_STAGE1_MODEL.name, FACE_MODEL.name]
def run_fid_diversity():
    fid_frames = []
    for gap in GAPS:
        table = compute_fid_table(
            FULL_CLIP_DIR / f'gap{gap:02d}', MODEL_NAMES,
            evaluation_dir=EVALUATION_DIR,
            evaluator_checkpoint=EVALUATOR_CKPT,
            evaluator_config=EVALUATOR_CONFIG,
            evaluator_stats_dir=EVALUATOR_STATS_DIR,
            device=DEVICE, batch_size=64, diversity_times=300,
        ).reset_index()
        table['gap'] = gap
        fid_frames.append(table)
    fid_df = pd.concat(fid_frames, ignore_index=True).set_index(['gap', 'model'])
    fid_df.to_csv(OUT_DIR / 'fid_diversity_metrics.csv')
    display(fid_df.round(6))

## Full-Clip R@K Semantic Retrieval

In [ ]:
def run_retrieval_rk():
    retrieval_frames = []
    for gap in GAPS:
        table = compute_retrieval_table(
            FULL_CLIP_DIR / f'gap{gap:02d}', MODEL_NAMES,
            evaluation_dir=EVALUATION_DIR,
            evaluator_checkpoint=EVALUATOR_CKPT,
            evaluator_config=EVALUATOR_CONFIG,
            evaluator_stats_dir=EVALUATOR_STATS_DIR,
            motion2text_json=MOTION2TEXT_JSON,
            text_model_path=EVALUATOR_TEXT_MODEL,
            device=DEVICE,
        ).reset_index()
        table['gap'] = gap
        retrieval_frames.append(table)
    retrieval_df = pd.concat(retrieval_frames, ignore_index=True).set_index(['gap', 'model'])
    retrieval_df.to_csv(OUT_DIR / 'retrieval_rk_metrics.csv')
    display(retrieval_df.round(4))

## Shared Validation Census

In [ ]:
split_names = read_split_file(EVAL_SPLIT)
body_available = set(discover_names(BODY_TOKEN_DIR, AUDIO_DIR, split_names))
face_available = set(discover_names(FACE_TOKEN_DIR, AUDIO_DIR, split_names))
common_names = [name for name in split_names if name in body_available and name in face_available]
if MAX_EVAL_CLIPS is not None:
    common_names = common_names[:MAX_EVAL_CLIPS]
print('listed validation clips:', len(split_names))
print('body-token clips:', len(body_available))
print('face-token clips:', len(face_available))
print('paired clips:', len(common_names))

In [ ]:
body_codecs = load_part_codecs(BODY_CHECKPOINTS, DEVICE, part_order=PART_ORDER)
face_codecs = load_part_codecs(FACE_CHECKPOINTS, DEVICE, part_order=MULTIMODAL_PART_ORDER)

def load_paired(token_dir, part_order, codecs):
    codebook_size = next(iter(codecs.values())).codebook_size
    quantizers = next(iter(codecs.values())).num_quantizers
    sequences, stats = load_sequences(
        common_names, token_dir, AUDIO_DIR,
        codebook_size=codebook_size,
        num_tokens_per_frame=len(part_order) * quantizers,
        audio_fps=10.0,
        source_motion_fps_fallback=20.0,
        motion_token_fps_override=None,
        motion_token_unit_length_override=None,
    )
    return sequences, stats

body_sequences, body_stats = load_paired(BODY_TOKEN_DIR, PART_ORDER, body_codecs)
face_sequences, face_stats = load_paired(FACE_TOKEN_DIR, MULTIMODAL_PART_ORDER, face_codecs)
assert [x['name'] for x in body_sequences] == [x['name'] for x in face_sequences]
print('body:', body_stats)
print('body+face:', face_stats)

## Face Codec Floor

These errors compare the decoded ground-truth face tokens against raw ARKit coefficients. They are the lower bound for any infiller using this codec.

In [ ]:
codec_face = evaluate_face_codec_reconstruction(
    face_sequences, face_codecs, FACE_DIR, DEVICE, part_order=MULTIMODAL_PART_ORDER
)
codec_columns = [
    'face_mae', 'face_rmse', 'face_velocity_rmse',
    'face_lip_rmse', 'face_lip_velocity_rmse',
    'face_non_lip_rmse', 'face_non_lip_velocity_rmse',
]
display(codec_face[codec_columns].mean().to_frame('codec_floor').round(6))
OUT_DIR.mkdir(parents=True, exist_ok=True)
codec_face.to_csv(OUT_DIR / 'face_codec_reconstruction.csv', index=False)

## Paired Infill Evaluation

Body RMSE is directly comparable across both models. Face metrics exist only for the body+face model and are measured against the face codec target, so interpret them above the codec floor.

In [ ]:
runs = [
    (BODY_MODEL, body_sequences, body_codecs, PART_ORDER),
    (FACE_STAGE1_MODEL, face_sequences, face_codecs, MULTIMODAL_PART_ORDER),
    (FACE_MODEL, face_sequences, face_codecs, MULTIMODAL_PART_ORDER),
]
frames = []
for spec, sequences, codecs, part_order in runs:
    print('loading', spec.name, spec.checkpoint)
    model = load_audio_motion_transformer(spec.checkpoint, DEVICE)
    frames.append(evaluate_model_windows(
        model, spec, sequences, codecs, GAPS,
        batch_size=BATCH_SIZE, device=DEVICE,
        windows_per_clip=WINDOWS_PER_CLIP, seed=42,
        slot_constrained=False, part_order=part_order,
    ))
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
window_df = pd.concat(frames, ignore_index=True)
window_df.to_csv(OUT_DIR / 'paired_window_metrics.csv', index=False)
print('rows:', len(window_df))

In [ ]:
by_gap, macro = summarize_window_metrics(window_df)
body_columns = [
    'body_rmse', 'body_velocity_rmse',
    'part_upper_acc', 'part_lower_acc', 'part_feet_acc', 'part_hands_acc',
]
display(by_gap[[c for c in body_columns if c in by_gap.columns]].round(5))
face_columns = [
    'part_face_acc', 'face_rmse', 'face_velocity_rmse',
    'face_lip_rmse', 'face_lip_velocity_rmse',
    'face_non_lip_rmse', 'face_non_lip_velocity_rmse',
]
face_by_gap = (
    window_df[window_df['model'].isin([FACE_STAGE1_MODEL.name, FACE_MODEL.name])]
    .groupby(['model', 'gap'])[[c for c in face_columns if c in window_df.columns]]
    .mean()
)
display(face_by_gap.round(5))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for model_name, group in by_gap.reset_index().groupby('model'):
    axes[0].plot(group['gap'], group['body_rmse'], marker='o', label=model_name)
    axes[1].plot(group['gap'], group['body_velocity_rmse'], marker='o', label=model_name)
for model_name, group in face_by_gap.reset_index().groupby('model'):
    axes[2].plot(group['gap'], group['face_lip_rmse'], marker='o', label=f'{model_name}: lip')
    axes[2].plot(group['gap'], group['face_non_lip_rmse'], marker='x', linestyle='--', label=f'{model_name}: non-lip')
axes[0].set_title('Body RMSE')
axes[1].set_title('Body velocity RMSE')
axes[2].set_title('Generated face RMSE')
for axis in axes:
    axis.set_xlabel('Gap token frames')
    axis.set_ylabel('Lower is better')
    axis.grid(alpha=0.25)
    axis.legend()
fig.tight_layout()
plt.show()

## Tiled Full-Clip Body and Face Visualizations

This diagnostic selects one Stage 2 clip nearest the median aggregate face RMSE across gaps 3, 7, and 15. The same complete clip is then tiled anchor-to-anchor at every gap setting, matching the full-clip evaluation protocol. `Raw GT`, `Codec GT`, Stage 1, and Stage 2 play for the complete usable duration. The 2D face is a diagnostic glyph driven directly by the 51 ARKit coefficients, not SuSu's mesh. Red timeline regions are generated interiors; grey frames are retained anchors or the uncovered clip tail. MP4 outputs include the original audio.

In [ ]:
VISUALIZATION_DIR = OUT_DIR / 'tiled_full_clip_visualizations'

def run_full_clip_visualizations():
    metrics_path = OUT_DIR / 'paired_window_metrics.csv'
    if 'window_df' in globals():
        metrics_frame = window_df
    elif metrics_path.exists():
        metrics_frame = pd.read_csv(metrics_path)
        print('loaded window metrics:', metrics_path)
    else:
        raise FileNotFoundError('Run the paired infill evaluation first: ' + str(metrics_path))
    selected = select_representative_clip_row(
        metrics_frame,
        model_name=FACE_MODEL.name,
        gaps=VISUALIZATION_GAPS,
        metric='face_rmse',
    )
    VISUALIZATION_DIR.mkdir(parents=True, exist_ok=True)
    selected.to_csv(VISUALIZATION_DIR / 'selected_clip.csv', index=False)
    display(selected)
    sequence_idx = int(selected.iloc[0]['sequence_idx'])
    item = face_sequences[sequence_idx]
    assert str(item['name']) == str(selected.iloc[0]['name'])
    audio_path = WAV_DIR / f"{item['name']}.wav"
    print('selected full clip:', item['name'])
    print('audio:', audio_path, audio_path.exists())

    stage1 = load_audio_motion_transformer(FACE_STAGE1_MODEL.checkpoint, DEVICE)
    stage2 = load_audio_motion_transformer(FACE_MODEL.checkpoint, DEVICE)
    visual_models = {
        'Stage 1': (stage1, FACE_STAGE1_MODEL),
        'Stage 2': (stage2, FACE_MODEL),
    }
    comparisons = {}
    try:
        for gap in VISUALIZATION_GAPS:
            print(f'generating complete clip at gap={gap}')
            comparison = prepare_tiled_full_clip_comparison(
                item, visual_models, face_codecs, gap=gap,
                device=DEVICE, template_bvh=TEMPLATE_BVH,
                motion_dir=MOTION_DIR, face_dir=FACE_DIR,
                batch_size=BATCH_SIZE, fps=20,
            )
            comparisons[int(gap)] = comparison
            png_path = VISUALIZATION_DIR / f'full_clip_gap{int(gap):02d}.png'
            mp4_path = VISUALIZATION_DIR / f'full_clip_gap{int(gap):02d}.mp4'
            fig = plot_tiled_full_clip_summary(
                comparison, keyframes=7, output_path=png_path
            )
            display(fig)
            plt.close(fig)
            save_tiled_full_clip_video(
                comparison, mp4_path, audio_path=audio_path,
                frame_step=VISUALIZATION_FRAME_STEP,
            )
            display(Video(filename=str(mp4_path), embed=False, html_attributes='controls'))
            print('saved:', png_path)
            print('saved:', mp4_path)
    finally:
        del stage1, stage2
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return selected, comparisons

if RUN_FULL_CLIP_VISUALIZATIONS:
    selected_visual_clip, full_clip_visual_comparisons = run_full_clip_visualizations()
else:
    print('Set RUN_FULL_CLIP_VISUALIZATIONS=True after loading face_sequences and face_codecs.')

## Run Full-Clip Metrics

Enable the flags in the configuration cell. Export must run once before FID or retrieval; afterward leave export disabled when rerunning metrics.

In [ ]:
if RUN_FULL_CLIP_EXPORT:
    run_full_clip_export()
if RUN_FID_DIVERSITY:
    run_fid_diversity()
if RUN_RETRIEVAL_RK:
    run_retrieval_rk()
if not any([RUN_FULL_CLIP_EXPORT, RUN_FID_DIVERSITY, RUN_RETRIEVAL_RK]):
    print('Enable the desired full-clip flags in the configuration cell.')

## Decision Rule

Accept the combined model only if face errors remain reasonably close to the codec floor and body RMSE does not regress materially on the paired subset. Motion FID/R@K should still be rerun with the existing full-clip evaluator before reporting the final model.